# Audio Event Extraction Pipeline — GLAP + CED + MiDashengLM (Kaggle T4×2)

Nhánh **audio** cho video retrieval: bắt **sự kiện âm thanh non-speech** (piano, còi xe, nhạc...) mà ASR không bắt được. Bù cho nhánh speech để giải query kiểu *"một người nói cảm ơn và bên cạnh có tiếng piano"*.

**Luồng** (window 5s, hop 2.5s):
- **GLAP** → embedding audio-text đa ngôn ngữ (1024-d) → xuất `.glap.npy` (cosine với text query).
- **CED** → tag AudioSet 527 nhãn ("Piano","Music","Siren"...) + score.
- **MiDashengLM** (int4) → caption, **chỉ** trên window có sự kiện non-speech (gate bằng CED). Bọc try/except: lỗi thì tự tắt, GLAP+CED vẫn chạy.

**Output** (khớp schema speech để fusion): `<id>.audio.json` + `<id>.glap.npy`.

**Trước khi chạy:** bật **Internet**, chọn **GPU T4 × 2**, sửa `DATASET_GLOB`. Checkpoint: skip video đã có `.audio.json`.

> Khác nhánh speech: KHÔNG cần convert model / cuDNN wrapper. Model tải thẳng từ HuggingFace.
> GLAP/MiDashengLM là code nghiên cứu — nếu API báo sai, sửa trong `audio_pipeline.py` (đã ghi chú).


## 1. Config

In [ ]:
DATASET_GLOB  = "/kaggle/input/**/*.mp4"   # ĐỆ QUY mọi dataset/thư mục con (sửa nếu cần)
WORK_DIR      = "/kaggle/working/audio_out"

NUM_GPUS      = 2          # T4×2
BATCH_SIZE    = 32         # batch GLAP/CED (encoder nhẹ, có thể nâng nếu VRAM dư)
USE_CAPTION   = True       # MiDashengLM caption (gate bằng CED). Tắt = chỉ GLAP+CED, nhanh hơn nhiều.
CAPTION_QUANT = "int4"     # "int4" (vừa T4, nhanh) hoặc "fp16"

import os
os.makedirs(WORK_DIR, exist_ok=True)
print("Config OK ->", WORK_DIR)


## 2. Ghi file pipeline

In [ ]:
%%writefile audio_pipeline.py
"""
Audio Event Extraction Pipeline cho Video Retrieval (NHÁNH AUDIO — bù cho ASR)
Bắt sự kiện âm thanh non-speech (piano, còi xe, nhạc...) mà ASR không bao giờ bắt được.

3 tầng đặc trưng / mỗi window (5s, hop 2.5s):
  (A) GLAP    : embedding audio-text đa ngôn ngữ (Dasheng + text enc) -> vector 1024-d
                -> retrieval mở: cosine(text_query, audio_window). XUẤT .glap.npy
  (B) CED     : tagging AudioSet 527 nhãn ("Piano","Siren","Music"...) -> tag rời + score
  (C) caption : MiDashengLM (int4) CHỈ trên window có sự kiện non-speech (GATE bằng CED)
                -> câu mô tả giàu nghĩa để index BM25. Bọc try/except: lỗi thì tự tắt,
                   GLAP+CED vẫn chạy bình thường.

Multi-GPU: --num_gpus 2 chia video round-robin, mỗi worker ghim 1 GPU.
Checkpoint: skip video đã có .audio.json.
Output mỗi video:
  <id>.audio.json : [{video_id,start,end,tags[],caption?,glap_idx}]   (khớp schema speech để fusion)
  <id>.glap.npy   : float16 [N_window x 1024], hàng i <-> glap_idx i

LƯU Ý: API GLAP/MiDashengLM là code nghiên cứu -> nếu Kaggle báo sai tên hàm/tham số,
chỉnh trong _load_* / _glap_batch / _caption (đã ghi chú rõ chỗ cần sửa).
"""

import os
import json
import subprocess
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import List, Optional, Tuple

import numpy as np


# ============================== CONFIG ==============================
GLAP_MODEL = "mispeech/GLAP"
CED_MODEL = "mispeech/ced-base"
CAPTION_MODEL = "mispeech/midashenglm-7b"

SR = 16000                 # cả GLAP/Dasheng, CED, MiDashengLM đều 16kHz mono
WINDOW_SEC = 5.0
HOP_SEC = 2.5
DEVICE = "cuda"

TOPK_TAGS = 5              # số nhãn AudioSet lưu / window
CAPTION_GATE = 0.30       # caption nếu tag non-speech mạnh nhất >= ngưỡng này
CAPTION_PROMPT = "Describe the audio, focusing on non-speech sounds, music and instruments."

# Top-1 tag thuộc nhóm này -> KHÔNG caption (chỉ caption window có SỰ KIỆN âm thanh thật,
# không phải tiếng nói / im lặng / tiếng phòng). Gồm các nhãn speech + ambience hay gặp.
SPEECH_STOP = {"Speech", "Male speech, man speaking", "Female speech, woman speaking",
               "Conversation", "Narration, monologue", "Speech synthesizer",
               "Child speech, kid speaking", "Silence",
               "Inside, small room", "Inside, large room or hall", "Inside, public space"}


# ============================== DATA MODEL ==============================
@dataclass
class AudioWindow:
    video_id: str
    start: float
    end: float
    tags: List[dict] = field(default_factory=list)   # [{"label","score"}]
    glap_idx: int = -1                                 # hàng trong .glap.npy
    caption: Optional[str] = None


# ============================== TÁCH AUDIO ==============================
def extract_audio(video_path: str, out_wav: str) -> str:
    cmd = ["ffmpeg", "-y", "-i", video_path, "-vn", "-ac", "1", "-ar", str(SR),
           "-c:a", "pcm_s16le", out_wav]
    proc = subprocess.run(cmd, capture_output=True)
    if proc.returncode != 0:
        tail = proc.stderr.decode("utf-8", "ignore")[-800:]
        raise RuntimeError(f"ffmpeg lỗi khi tách audio '{video_path}':\n{tail}")
    return out_wav


def load_wav(wav_path: str) -> np.ndarray:
    import soundfile as sf
    wav, sr = sf.read(wav_path, dtype="float32")
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != SR:  # phòng hờ — ffmpeg đã resample về SR
        import librosa
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
    return wav


def make_windows(wav: np.ndarray) -> List[Tuple[float, float, np.ndarray]]:
    """Cắt window trượt (WINDOW_SEC, hop HOP_SEC). Window cuối zero-pad cho đủ độ dài."""
    win = int(WINDOW_SEC * SR)
    hop = int(HOP_SEC * SR)
    n = len(wav)
    out = []
    start = 0
    while start < max(1, n):
        seg = wav[start:start + win]
        real_end = min(start + len(seg), n)
        if len(seg) < win:
            seg = np.pad(seg, (0, win - len(seg)))
        out.append((start / SR, real_end / SR, seg))
        if start + win >= n:
            break
        start += hop
    return out


# ============================== EXTRACTOR (load model 1 lần / process) ==============================
class AudioExtractor:
    def __init__(self, use_caption: bool = True, device: str = DEVICE,
                 batch_size: int = 32, caption_quant: str = "int4"):
        import torch
        self.torch = torch
        self.device = device
        self.batch_size = batch_size
        self.use_caption = use_caption
        self._cap_warned = False

        # ---- (A) GLAP (audio-text embedding) ----
        # API tài liệu: glap_model.glap_inference() -> .encode_audio(tensor[B,T]) / .encode_text([str])
        from glap_model import glap_inference
        self.glap = glap_inference()
        try:
            self.glap = self.glap.to(device).eval()
        except Exception:
            pass  # vài bản tự đặt device

        # ---- (B) CED (AudioSet 527-tag) ----
        from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
        self.ced = AutoModelForAudioClassification.from_pretrained(
            CED_MODEL, trust_remote_code=True).to(device).eval()
        self.ced_fe = AutoFeatureExtractor.from_pretrained(CED_MODEL, trust_remote_code=True)
        self.id2label = self.ced.config.id2label

        # ---- (C) MiDashengLM caption (tùy chọn, bọc lỗi) ----
        self.cap_model = self.cap_proc = self.cap_tok = None
        if use_caption:
            try:
                self._load_caption(caption_quant)
            except Exception as e:
                print(f"[audio] CẢNH BÁO: không load được caption model ({e}); chạy không caption.",
                      flush=True)
                self.use_caption = False

    def _load_caption(self, quant: str):
        import torch
        from transformers import AutoModelForCausalLM, AutoProcessor, AutoTokenizer
        kw = dict(trust_remote_code=True)
        if quant == "int4":
            from transformers import BitsAndBytesConfig
            kw["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4")
            kw["device_map"] = {"": 0}
            kw["torch_dtype"] = torch.float16   # audio encoder (không bị quant) ở fp16
        else:
            kw["torch_dtype"] = torch.float16
        self.cap_model = AutoModelForCausalLM.from_pretrained(CAPTION_MODEL, **kw).eval()
        if quant != "int4":
            self.cap_model = self.cap_model.to(self.device)
        self.cap_proc = AutoProcessor.from_pretrained(CAPTION_MODEL, trust_remote_code=True)
        # decode bằng tokenizer riêng — processor KHÔNG có batch_decode.
        self.cap_tok = AutoTokenizer.from_pretrained(CAPTION_MODEL, trust_remote_code=True)

    # ---------- (A) GLAP ----------
    def _glap_batch(self, segs: List[np.ndarray]) -> np.ndarray:
        torch = self.torch
        vecs = []
        with torch.no_grad():
            for i in range(0, len(segs), self.batch_size):
                chunk = segs[i:i + self.batch_size]
                x = torch.from_numpy(np.stack(chunk)).float().to(self.device)  # [B, T]
                emb = self.glap.encode_audio(x)
                emb = emb.float()
                emb = emb / emb.norm(dim=-1, keepdim=True).clamp_min(1e-8)     # L2-norm cho cosine
                vecs.append(emb.cpu().numpy())
        return np.concatenate(vecs, axis=0).astype(np.float16)

    def encode_text(self, texts: List[str]) -> np.ndarray:
        """Tiện dụng lúc query: encode 1 hoặc nhiều câu text -> vector chuẩn hóa."""
        torch = self.torch
        with torch.no_grad():
            emb = self.glap.encode_text(list(texts)).float()
            emb = emb / emb.norm(dim=-1, keepdim=True).clamp_min(1e-8)
        return emb.cpu().numpy().astype(np.float16)

    # ---------- (B) CED ----------
    def _ced_batch(self, segs: List[np.ndarray]) -> List[List[dict]]:
        torch = self.torch
        out: List[List[dict]] = []
        with torch.no_grad():
            for i in range(0, len(segs), self.batch_size):
                chunk = segs[i:i + self.batch_size]
                inputs = self.ced_fe(chunk, sampling_rate=SR, return_tensors="pt")
                inputs = {k: v.to(self.device) for k, v in inputs.items()}
                logits = self.ced(**inputs).logits           # [B, 527]
                probs = torch.sigmoid(logits)                # multi-label
                topv, topi = probs.topk(TOPK_TAGS, dim=-1)
                for v_row, i_row in zip(topv.cpu().tolist(), topi.cpu().tolist()):
                    out.append([{"label": self.id2label[int(idx)], "score": round(float(sc), 4)}
                                for sc, idx in zip(v_row, i_row)])
        return out

    # ---------- (C) caption ----------
    @staticmethod
    def _should_caption(tags: List[dict]) -> bool:
        # Gate theo TOP-1: chỉ caption khi nhãn MẠNH NHẤT là sự kiện non-speech. Window bản tin
        # top-1 = Speech -> bỏ qua; window có nhạc/sự kiện top-1 = Music/... -> caption.
        # Giữ caption HIẾM + đúng chỗ -> không làm vỡ ước lượng thời gian.
        if not tags:
            return False
        top = tags[0]
        return top["label"] not in SPEECH_STOP and top["score"] >= CAPTION_GATE

    def _caption(self, seg: np.ndarray) -> Optional[str]:
        if not self.use_caption or self.cap_model is None:
            return None
        try:
            # API đúng theo model card mispeech/midashenglm-7b:
            #   apply_chat_template(...).to(device, dtype) tự lo dtype; decode bằng TOKENIZER.
            messages = [{"role": "user", "content": [
                {"type": "text", "text": CAPTION_PROMPT},
                {"type": "audio", "audio": seg}]}]
            model_inputs = self.cap_proc.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=True,
                add_special_tokens=True, return_dict=True,
            ).to(device=self.cap_model.device, dtype=self.cap_model.dtype)
            with self.torch.no_grad():
                gen = self.cap_model.generate(**model_inputs, max_new_tokens=64, do_sample=False)
            text = self.cap_tok.batch_decode(gen, skip_special_tokens=True)[0]
            return text.strip()
        except Exception as e:
            if not self._cap_warned:
                print(f"[audio] CẢNH BÁO: caption lỗi ({e}); tắt caption phần còn lại.", flush=True)
                self._cap_warned = True
                self.use_caption = False
            return None

    # ---------- pipeline 1 video ----------
    def process(self, wav: np.ndarray, video_id: str) -> Tuple[List[AudioWindow], np.ndarray]:
        wins = make_windows(wav)
        segs = [s for _, _, s in wins]
        glap_vecs = self._glap_batch(segs)        # [N, 1024] fp16
        all_tags = self._ced_batch(segs)          # [N][topk]

        out: List[AudioWindow] = []
        for idx, ((start, end, seg), tags) in enumerate(zip(wins, all_tags)):
            cap = self._caption(seg) if self._should_caption(tags) else None
            out.append(AudioWindow(video_id, round(start, 3), round(end, 3),
                                   tags=tags, glap_idx=idx, caption=cap))
        return out, glap_vecs


# ============================== XỬ LÝ 1 VIDEO ==============================
def process_video(extractor: AudioExtractor, video_path: str, work_dir: str) -> int:
    video_id = Path(video_path).stem
    wav_path = os.path.join(work_dir, f"{video_id}.audio.wav")
    extract_audio(video_path, wav_path)
    try:
        wav = load_wav(wav_path)
        windows, glap_vecs = extractor.process(wav, video_id)

        np.save(os.path.join(work_dir, f"{video_id}.glap.npy"), glap_vecs)
        with open(os.path.join(work_dir, f"{video_id}.audio.json"), "w", encoding="utf-8") as f:
            json.dump([asdict(w) for w in windows], f, ensure_ascii=False, indent=2)
        return len(windows)
    finally:
        try:
            os.remove(wav_path)
        except OSError:
            pass


# ============================== WORKER ĐA GPU ==============================
def _worker(gpu_id: int, videos: List[str], work_dir: str, use_caption: bool,
            batch_size: int, caption_quant: str):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_id)  # PHẢI set trước khi torch init
    extractor = AudioExtractor(use_caption=use_caption, device="cuda",
                               batch_size=batch_size, caption_quant=caption_quant)
    for v in videos:
        vid = Path(v).stem
        if os.path.exists(os.path.join(work_dir, f"{vid}.audio.json")):
            print(f"[gpu{gpu_id}] skip {vid} (đã có)", flush=True)
            continue
        try:
            n = process_video(extractor, v, work_dir)
            print(f"[gpu{gpu_id}] {vid} -> {n} windows", flush=True)
        except Exception as e:
            print(f"[gpu{gpu_id}] LỖI {vid}: {e}", flush=True)


def run_multi_gpu(videos, num_gpus, work_dir, use_caption, batch_size, caption_quant):
    import multiprocessing as mp
    shards = [videos[i::num_gpus] for i in range(num_gpus)]
    ctx = mp.get_context("spawn")  # bắt buộc cho CUDA
    procs = []
    for gid in range(num_gpus):
        if not shards[gid]:
            continue
        p = ctx.Process(target=_worker, args=(gid, shards[gid], work_dir, use_caption,
                                              batch_size, caption_quant))
        p.start(); procs.append(p)
    for p in procs:
        p.join()


# ============================== MAIN ==============================
if __name__ == "__main__":
    import argparse
    ap = argparse.ArgumentParser(description="Audio event extraction (GLAP + CED + MiDashengLM)")
    ap.add_argument("--videos", nargs="+", required=True)
    ap.add_argument("--num_gpus", type=int, default=1)
    ap.add_argument("--work_dir", default="./audio_out")
    ap.add_argument("--batch_size", type=int, default=32, help="batch GLAP/CED")
    ap.add_argument("--no_caption", action="store_true", help="tắt MiDashengLM captioning")
    ap.add_argument("--caption_quant", default="int4", choices=["int4", "fp16"])
    args = ap.parse_args()

    os.makedirs(args.work_dir, exist_ok=True)
    use_caption = not args.no_caption

    if args.num_gpus > 1:
        run_multi_gpu(args.videos, args.num_gpus, args.work_dir, use_caption,
                      args.batch_size, args.caption_quant)
    else:
        extractor = AudioExtractor(use_caption=use_caption, batch_size=args.batch_size,
                                   caption_quant=args.caption_quant)
        for v in args.videos:
            vid = Path(v).stem
            if os.path.exists(os.path.join(args.work_dir, f"{vid}.audio.json")):
                print(f"skip {vid} (đã có)"); continue
            try:
                n = process_video(extractor, v, args.work_dir)
                print(f"[{vid}] -> {n} windows")
            except Exception as e:
                print(f"LỖI {vid}: {e}")


## 3. Setup
Cài lib (không cần convert model). MiDashengLM tải từ HF lúc chạy lần đầu.

In [ ]:
# === Setup nhánh audio — KHÔNG -q để THẤY lỗi. ===
# GLAP: thử PyPI, fail thì cài từ GitHub source.
!pip install glap_model || pip install "git+https://github.com/xiaomi-research/dasheng-glap.git"
!pip install soundfile librosa accelerate bitsandbytes
# QUAN TRỌNG: transformers 5.0 của Kaggle phá custom code CED + MiDashengLM.
# Pin 4.53.3 = <5.0 (CED chạy ĐÚNG) nhưng >=4.52 (Qwen2.5-Omni cho MiDashengLM). GLAP ok cả 2.
!pip install "transformers==4.53.3"

!python -c "from glap_model import glap_inference; print('>>> glap_model OK')"
!python -c "from transformers import AutoModelForAudioClassification, AutoFeatureExtractor; print('>>> CED deps OK')"
!python -c "import torch, transformers, numpy as np; print('torch', torch.__version__, '| transformers', transformers.__version__, '| numpy', np.__version__)"
print(">>> Setup xong. PHẢI Restart kernel (đã đổi transformers) rồi chạy lại từ đây.")


## (tùy chọn) Ước lượng tổng thời gian
Benchmark 2-lần-chạy để tách thời gian load model khỏi throughput thật. Caption làm thời gian biến thiên theo tỉ lệ window có sự kiện — chạy với đúng `USE_CAPTION` để ước lượng sát.

In [ ]:
import os, glob, time, shutil, subprocess
from concurrent.futures import ThreadPoolExecutor

videos = sorted(glob.glob(DATASET_GLOB, recursive=True))
assert videos, "Không thấy video — kiểm tra DATASET_GLOB."

def ffdur(p):
    out = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1", p],
        capture_output=True, text=True).stdout.strip()
    return float(out) if out else 0.0

print(f"Quét độ dài {len(videos)} video...")
with ThreadPoolExecutor(max_workers=16) as ex:
    durs = list(ex.map(ffdur, videos))
total_audio_s = sum(durs)
order = sorted(range(len(videos)), key=lambda i: durs[i])

# Audio branch RẤT nhanh -> proc(1 video) < nhiễu load. Đo trên NHIỀU video cho ổn định:
#   runA = [A ngắn]            -> tA = load + proc(A)
#   runB = [A] + B_SET (12 video quanh median) -> tB = tA + proc(B_SET)
A = videos[order[int(0.10 * (len(order) - 1))]]
mid = len(order) // 2
B_set = [videos[i] for i in order[max(0, mid - 6):mid + 6]]
durA = ffdur(A)
durB_total = sum(ffdur(v) for v in B_set)
print(f"A (đo load): {os.path.basename(A)}  {durA/60:.1f} phút")
print(f"B_SET: {len(B_set)} video quanh median, tổng {durB_total/60:.1f} phút")

def bench(vids, d):
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
    cmd = ["python", "audio_pipeline.py", "--videos", *vids, "--num_gpus", "1",
           "--work_dir", d, "--batch_size", str(BATCH_SIZE), "--caption_quant", CAPTION_QUANT]
    if not USE_CAPTION:
        cmd.append("--no_caption")
    t = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:]); print(r.stderr[-2000:]); raise RuntimeError("benchmark lỗi")
    return time.time() - t

print(">>> warm-up (tải/cache model, KHÔNG tính giờ)..."); bench([A], "/kaggle/working/_abWarm")
print(">>> runA (load + A)..."); tA = bench([A], "/kaggle/working/_abA"); print(f"    tA = {tA:.1f}s")
print(f">>> runB (load + A + {len(B_set)} video)..."); tB = bench([A] + B_set, "/kaggle/working/_abB"); print(f"    tB = {tB:.1f}s")

delta = tB - tA                       # ~ thời gian xử lý B_SET
assert delta > 0.5, "delta quá nhỏ — audio branch quá nhanh; tăng số video trong B_set."
xrt = durB_total / delta
load_s = max(0.0, tA - durA / xrt)
per_gpu_audio = total_audio_s / NUM_GPUS
est_wall_s = load_s + per_gpu_audio / xrt
n_sessions = est_wall_s / 3600 / 11.0

def hm(s):
    m = int(s // 60); h, m = divmod(m, 60); return f"{h}h{m:02d}m"

print("\n===== ƯỚC LƯỢNG =====")
print(f"Tổng audio          : {total_audio_s/3600:.1f} giờ ({len(videos)} video)")
print(f"Tốc độ (1 GPU)      : {xrt:.1f}x realtime  (caption={USE_CAPTION}, batch={BATCH_SIZE}, {CAPTION_QUANT})")
print(f"Load model (1 lần)  : {load_s:.0f}s")
print(f"Ước lượng wall ({NUM_GPUS} GPU): {hm(est_wall_s)}")
print(f"Số session 12h cần  : ~{n_sessions:.2f}")

for d in ["/kaggle/working/_abWarm", "/kaggle/working/_abA", "/kaggle/working/_abB"]:
    shutil.rmtree(d, ignore_errors=True)


## 4. Run
Chạy đa GPU. Glob đệ quy. Checkpoint skip `.audio.json`.

In [ ]:
import glob, subprocess
videos = sorted(glob.glob(DATASET_GLOB, recursive=True))
print(f"Tìm thấy {len(videos)} video từ {DATASET_GLOB}")
assert videos, "Không thấy video nào — kiểm tra lại DATASET_GLOB."

cmd = ["python", "audio_pipeline.py",
       "--videos", *videos,
       "--num_gpus", str(NUM_GPUS),
       "--work_dir", WORK_DIR,
       "--batch_size", str(BATCH_SIZE),
       "--caption_quant", CAPTION_QUANT]
if not USE_CAPTION:
    cmd.append("--no_caption")
print(" ".join(cmd[:2]), "...", len(videos), "videos | caption =", USE_CAPTION)
subprocess.run(cmd, check=True)


## 5. Kiểm tra output
Xác nhận: số window, **tỉ lệ window được caption** (gate có hợp lý không), **tag phổ biến** (CED có bắt đúng sự kiện không), và shape `.glap.npy`.

In [ ]:
import json, glob, os, collections
import numpy as np

files = sorted(glob.glob(os.path.join(WORK_DIR, "*.audio.json")))
print(f"{len(files)} file .audio.json")

n_win = n_cap = 0
top1 = collections.Counter()
for f in files:
    ws = json.load(open(f, encoding="utf-8"))
    n_win += len(ws)
    for w in ws:
        if w.get("caption"):
            n_cap += 1
        if w["tags"]:
            top1[w["tags"][0]["label"]] += 1

print(f"Tổng window: {n_win} | có caption: {n_cap} ({100*n_cap/max(1,n_win):.1f}%)")
print("Top-1 tag phổ biến:")
for label, c in top1.most_common(15):
    print(f"   {c:6d}  {label}")

npys = sorted(glob.glob(os.path.join(WORK_DIR, "*.glap.npy")))
if npys:
    a = np.load(npys[0])
    print(f"\nglap npy ví dụ: {os.path.basename(npys[0])}  shape={a.shape}  dtype={a.dtype}")

# Mẫu vài window (ưu tiên window có caption)
if files:
    ws = json.load(open(files[0], encoding="utf-8"))
    print(f"\n--- Mẫu: {os.path.basename(files[0])} ---")
    shown = 0
    for w in ws:
        if w.get("caption") or shown < 5:
            tg = ", ".join(f"{t['label']}({t['score']})" for t in w["tags"][:3])
            print(f"[{w['start']:6.1f}-{w['end']:6.1f}] {tg}")
            if w.get("caption"):
                print(f"            cap: {w['caption']}")
            shown += 1
        if shown >= 10:
            break
